Installing CodeCarbon

In [1]:
!pip install CodeCarbon

Knowledge Distillation (Sequence) Step 2: Student Training

In [2]:
import os
import csv
import glob
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import OfflineEmissionsTracker

student_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # tokenizer source (training never changes it)
model_dir = "/content/distilled_model"
version = "distilled_tinyllama"
country = "USA"        # must match the other three scripts
num_runs = 5
max_new_tokens = 300

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")
hardware = torch.cuda.get_device_name(0)

# Load only if this exact model + version isn't already in memory
if globals().get("loaded_config") != (model_dir, version):
    if "model" in globals():
        raise RuntimeError("A different model/version is loaded. Restart the session first.")
    tokenizer = AutoTokenizer.from_pretrained(student_name)
    model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16, device_map="cuda")
    model.eval()
    loaded_config = (model_dir, version)
else:
    print("Model already loaded, skipping")

# Model size = storage size of the saved student weights
size_gb = sum(os.path.getsize(f) for f in glob.glob(os.path.join(model_dir, "*.safetensors"))) / 1e9

print("Version:", version)
print("Hardware:", hardware)
print("Dtype:", model.dtype)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")
print(f"Model size on disk: {size_gb:.2f} GB")
print(f"GPU memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

prompt = ("Generate Google-style docstring for the following Python function: {code}. "
          "Only output the docstring and function code.").format(code=Code)   # P-2

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)
input_tokens = inputs["input_ids"].shape[1]

results_csv = f"results_{version}.csv"
outputs_file = f"docstrings_{version}.txt"

# Warm up once (untimed, untracked)
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

with open(results_csv, "w", newline="") as f, open(outputs_file, "w") as out_f:
    writer = csv.writer(f)
    writer.writerow(["run", "input_tokens", "output_tokens", "latency_s",
                     "emissions_kg", "energy_kwh", "accuracy_PF"])

    for run in range(1, num_runs + 1):
        # Fresh tracker per run, wrapped around generation only
        tracker = OfflineEmissionsTracker(country_iso_code=country, measure_power_secs=1,
                                          save_to_file=False, log_level="error")
        tracker.start()
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                           do_sample=True, temperature=0.7,
                                           eos_token_id=tokenizer.eos_token_id,
                                           pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        latency = time.perf_counter() - start
        emissions_kg = tracker.stop()
        energy_kwh = tracker.final_emissions_data.energy_consumed

        output_tokens = generated_ids.shape[1] - input_tokens
        text = tokenizer.decode(generated_ids[0][input_tokens:], skip_special_tokens=True)
        out_f.write(f"--- Run {run} ---\n{text}\n\n")

        writer.writerow([run, input_tokens, output_tokens, f"{latency:.3f}",
                         f"{emissions_kg:.4e}", f"{energy_kwh:.4e}", ""])
        f.flush()

        print(f"Run {run}: {latency:.3f}s | {output_tokens} tokens | "
              f"{emissions_kg:.4e} kg CO2eq | {energy_kwh:.4e} kWh")

print(f"Docstrings saved to {outputs_file}")
print(f"Metrics saved to {results_csv}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Version: distilled_tinyllama
Hardware: Tesla T4
Dtype: torch.float16
Parameters: 1.10 B
Model size on disk: 2.20 GB
GPU memory footprint: 2.20 GB


[codecarbon INFO @ 00:52:00] offline tracker init
[codecarbon WARNING @ 00:52:00] Multiple instances of codecarbon are allowed to run at the same time.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: 15.343s | 242 tokens | 8.6262e-05 kg CO2eq | 2.3347e-04 kWh


...

Run 2: 7.233s | 218 tokens | 5.0856e-05 kg CO2eq | 1.3764e-04 kWh


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 3: 6.804s | 223 tokens | 5.1949e-05 kg CO2eq | 1.4060e-04 kWh


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 4: 7.745s | 252 tokens | 5.9318e-05 kg CO2eq | 1.6055e-04 kWh
Run 5: 6.632s | 222 tokens | 5.2050e-05 kg CO2eq | 1.4088e-04 kWh
Docstrings saved to docstrings_distilled_tinyllama.txt
Metrics saved to results_distilled_tinyllama.csv


Download Files

In [ ]:
from google.colab import files

files.download(results_csv)        # results_distilled_tinyllama.csv
files.download(outputs_file)       # docstrings_distilled_tinyllama.txt